# Exercise 3: Break Identifiability

Lab 23 shows that when two channels' spend moves together, the reference model cannot tell them apart: individual estimates swap around while the sum of their contributions stays stable.
This exercise asks you to reproduce that stress test yourself: make `meta` and `youtube` spend collinear, refit the reference model, and check that the individual split becomes unstable while the combined estimate does not.

In [1]:
import arviz as az
import numpy as np
import pymc as pm

from mmm_book import RANDOM_SEED
from mmm_book.plotting import mmm_style
from mmm_book.reference import (
    REVENUE_SCALE,
    build_features,
    national_aggregate,
    reference_model,
)
from mmm_book.sedge import generate_sedge

mmm_style()


def check(idata):
    n_divergent = int(idata.sample_stats["diverging"].sum())
    r_hat = float(az.summary(idata, kind="diagnostics")["r_hat"].max())
    print(f"divergent transitions: {n_divergent}, largest r_hat: {r_hat:.3f}")


_MEDIA_COL = {
    "tv": "tv_grps",
    "search": "search_spend",
    "meta": "meta_spend",
    "youtube": "youtube_spend",
    "ooh": "ooh_spend",
}

## The independent-spend fit

Sampling uses the settings every lab uses: four chains of 1,000 draws after 1,000 warm-up steps, with a target acceptance rate of 0.95.
The cell checks that the fit samples cleanly (no divergences, every `r_hat` at or below 1.01), and Lab 18 shows what goes wrong with fewer, shorter chains.
The model fixes each channel's adstock and saturation at the values the data were built with, so only the channel scales, baseline and controls are estimated.
The case-study notebooks estimate the transforms as well.
This independent-spend fit, on the national aggregate as recorded, is the case to compare against.
It is not a clean case: Meta and YouTube already share a paid-social budget, their features are correlated (about 0.84 on the recorded data, as Lab 23 prints), and Lab 23 found their split unreliable even here.
The question is how much worse forcing their spend together makes it.

In [2]:
df = generate_sedge()
nat = national_aggregate(df)
model = reference_model(nat)

with model:
    idata = pm.sample(
        draws=1000,
        tune=1000,
        chains=4,
        cores=1,
        target_accept=0.95,
        random_seed=RANDOM_SEED,
        progressbar=False,
    )

check(idata)

Initializing NUTS using jitter+adapt_diag...


Sequential sampling (4 chains in 1 job)


NUTS: [intercept, trend, season_a, season_b, omega_launch, delta_promo, gamma_log_price, gamma_distribution, gamma_stock, gamma_temperature, gamma_rainfall, gamma_category, beta_tv, beta_search, beta_meta, beta_youtube, beta_ooh, sigma]


Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 25 seconds.


divergent transitions: 0, largest r_hat: 1.004


## Task

Build `nat_collinear`, a copy of `nat` in which `meta_spend` and `youtube_spend` are both replaced by their shared average, so the two channels move together almost perfectly (as in Lab 23).
The two channels keep their own adstock alpha and Hill slope, so `build_features` will turn this shared spend into two highly correlated, but not identical, features.
Your `nat_collinear` should push the correlation between the resulting `meta` and `youtube` features, `feature_corr` below, above 0.9.

In [3]:
# YOUR CODE HERE: replace meta_spend and youtube_spend in nat_collinear with their shared
# average, so the two channels become collinear.
# Stub: leaves nat unchanged, so this notebook still runs, but the channels stay
# independent and the checks below will not pass yet.
nat_collinear = nat.copy()

feat_collinear = build_features(nat_collinear)
feature_corr = float(np.corrcoef(feat_collinear["meta"], feat_collinear["youtube"])[0, 1])
print(f"correlation between meta and youtube features: {feature_corr:.3f}")

correlation between meta and youtube features: 0.840


## Refitting on the collinear variant

Sampling settings are the same as above.

In [4]:
model_collinear = reference_model(nat_collinear)
with model_collinear:
    idata_collinear = pm.sample(
        draws=1000,
        tune=1000,
        chains=4,
        cores=1,
        target_accept=0.95,
        random_seed=RANDOM_SEED,
        progressbar=False,
    )

check(idata_collinear)

Initializing NUTS using jitter+adapt_diag...


Sequential sampling (4 chains in 1 job)


NUTS: [intercept, trend, season_a, season_b, omega_launch, delta_promo, gamma_log_price, gamma_distribution, gamma_stock, gamma_temperature, gamma_rainfall, gamma_category, beta_tv, beta_search, beta_meta, beta_youtube, beta_ooh, sigma]


Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 25 seconds.


divergent transitions: 0, largest r_hat: 1.004


## Comparing individual and combined ROAS

For each posterior draw, `beta_meta` and `beta_youtube` imply a per-channel ROAS.
The combined ROAS, Meta plus YouTube contribution over Meta plus YouTube spend, is what the data can still identify even when the individual split cannot.
This part is provided for you; only the construction of `nat_collinear` above is yours to implement.

In [5]:
def channel_roas_draws(nat_, idata_, ch):
    feat_ = build_features(nat_)
    beta_draws = idata_.posterior[f"beta_{ch}"].values.flatten()
    contribution_total = float(feat_[ch].sum()) * REVENUE_SCALE
    spend_total = float(nat_[_MEDIA_COL[ch]].to_numpy(dtype=float).sum())
    return beta_draws * contribution_total / spend_total


roas_meta_base = channel_roas_draws(nat, idata, "meta")
roas_youtube_base = channel_roas_draws(nat, idata, "youtube")
roas_meta_col = channel_roas_draws(nat_collinear, idata_collinear, "meta")
roas_youtube_col = channel_roas_draws(nat_collinear, idata_collinear, "youtube")


def combined_roas(rm, ry, n_):
    spend_m, spend_y = float(n_["meta_spend"].sum()), float(n_["youtube_spend"].sum())
    return (rm * spend_m + ry * spend_y) / (spend_m + spend_y)


combined_base = combined_roas(roas_meta_base, roas_youtube_base, nat)
combined_col = combined_roas(roas_meta_col, roas_youtube_col, nat_collinear)

p_meta_higher_base = float((roas_meta_base > roas_youtube_base).mean())
p_meta_higher_col = float((roas_meta_col > roas_youtube_col).mean())

print(f"P(meta > youtube), independent-spend fit: {p_meta_higher_base:.2f}")
print(f"P(meta > youtube), collinear fit:         {p_meta_higher_col:.2f}")
meta_sd_msg = f"{roas_meta_base.std():.3f} vs {roas_meta_col.std():.3f}"
combined_sd_msg = f"{combined_base.std():.3f} vs {combined_col.std():.3f}"
print(f"meta ROAS sd, independent-spend vs collinear fit: {meta_sd_msg}")
print(f"combined ROAS sd, independent-spend vs collinear fit: {combined_sd_msg}")

P(meta > youtube), independent-spend fit: 0.91
P(meta > youtube), collinear fit:         0.91
meta ROAS sd, independent-spend vs collinear fit: 0.824 vs 0.824
combined ROAS sd, independent-spend vs collinear fit: 0.409 vs 0.409


## Self-check

In [6]:
meta_sd_ratio = roas_meta_col.std() / max(roas_meta_base.std(), 1e-9)
combined_sd_ratio = combined_col.std() / max(combined_base.std(), 1e-9)

checks = {
    "feature correlation > 0.9": feature_corr > 0.9,
    "individual (meta) ROAS got less certain under collinearity": (
        roas_meta_col.std() > roas_meta_base.std()
    ),
    "combined ROAS stayed more stable than the individual split": (
        combined_sd_ratio < meta_sd_ratio
    ),
}
for name, ok in checks.items():
    print(f"{name}: {'PASS' if ok else 'FAIL'}")
print("Self-check: PASS" if all(checks.values()) else "Self-check: FAIL")

feature correlation > 0.9: FAIL
individual (meta) ROAS got less certain under collinearity: FAIL
combined ROAS stayed more stable than the individual split: FAIL
Self-check: FAIL
